In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSVs (files exist in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Ensure numeric types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# --- metric.csv summary ---
metric_row_count = metric_df.shape[0]
metric_unique_cmdb = pd.Series(sorted(metric_df['cmdb_id'].dropna().unique()))
metric_unique_kpi = pd.Series(sorted(metric_df['kpi_name'].dropna().unique()))
metric_ts_min = metric_df['ts_dt'].min()
metric_ts_max = metric_df['ts_dt'].max()

# Aggregate by (cmdb_id, kpi_name)
m_group = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
m_agg_basic = m_group.agg(count='count', min_value='min', max_value='max', mean_value='mean').reset_index()

# Quantiles computed on full series per group (global within file)
m_p95 = m_group.quantile(0.95).reset_index(name='p95')
m_p90 = m_group.quantile(0.90).reset_index(name='p90')
m_p50 = m_group.quantile(0.50).reset_index(name='p50')
m_p5  = m_group.quantile(0.05).reset_index(name='p5')

# Merge quantiles into aggregated df
metric_agg = m_agg_basic.merge(m_p95, on=['cmdb_id','kpi_name']) \
                        .merge(m_p90, on=['cmdb_id','kpi_name']) \
                        .merge(m_p50, on=['cmdb_id','kpi_name']) \
                        .merge(m_p5,  on=['cmdb_id','kpi_name'])

# Order by count desc and keep top 50
metric_agg_top50 = metric_agg.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# Keep column order tidy
metric_agg_top50 = metric_agg_top50[['cmdb_id','kpi_name','count','min_value','max_value','mean_value','p95','p90','p50','p5']]

# --- trace.csv summary ---
trace_row_count = trace_df.shape[0]
trace_unique_cmdb = pd.Series(sorted(trace_df['cmdb_id'].dropna().unique()))
trace_unique_trace = pd.Series(sorted(trace_df['trace_name'].dropna().unique()))
trace_ts_min = trace_df['ts_dt'].min()
trace_ts_max = trace_df['ts_dt'].max()

# Aggregate by (cmdb_id, trace_name)
t_group = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
t_agg_basic = t_group.agg(count='count', min_value='min', max_value='max', mean_value='mean').reset_index()

# Quantiles: P95, P50, P5
t_p95 = t_group.quantile(0.95).reset_index(name='p95')
t_p50 = t_group.quantile(0.50).reset_index(name='p50')
t_p5  = t_group.quantile(0.05).reset_index(name='p5')

trace_agg = t_agg_basic.merge(t_p95, on=['cmdb_id','trace_name']) \
                       .merge(t_p50, on=['cmdb_id','trace_name']) \
                       .merge(t_p5,  on=['cmdb_id','trace_name'])

# Order by count desc and keep up to 50 rows
trace_agg_top50 = trace_agg.sort_values('count', ascending=False).head(50).reset_index(drop=True)
trace_agg_top50 = trace_agg_top50[['cmdb_id','trace_name','count','min_value','max_value','mean_value','p95','p50','p5']]

# Final compact outputs (variables shown, not full raw frames)
metric_row_count, metric_unique_cmdb, metric_unique_kpi, metric_ts_min, metric_ts_max, metric_agg_top50, \
trace_row_count, trace_unique_cmdb, trace_unique_trace, trace_ts_min, trace_ts_max, trace_agg_top50

```
Out[1]:
```
```python
summary = (
    "Metric file (metric.csv):\n"
    "- Total rows: 91,076\n"
    "- Unique cmdb_id count: 56 (examples include db_001..redis_012)\n"
    "- Unique kpi_name count: 132 (various OS, JVM, network, DB KPIs)\n"
    "- Time range (Asia/Shanghai, UTC+8): 2020-05-26 01:30 to 2020-05-26 02:30\n"
    "- Aggregation summary (by cmdb_id, kpi_name): top groups sorted by count. "
    "Top group: cmdb_id=osb_001, kpi_name=app.num, count=61, min=399.0, max=635.0, mean≈524.82, p95=574.0, p90=562.0, p50=531.0, p5=461.0. "
    "Most of the top-50 aggregated groups have counts around 60–61.\n\n"
    "Trace file (trace.csv):\n"
    "- Total rows: 5,520\n"
    "- Unique cmdb_id count: 9 (examples: docker_001..docker_008, os_021, os_022)\n"
    "- Unique trace_name count: 72\n"
    "- Time range (Asia/Shanghai, UTC+8): 2020-05-26 02:00 to 2020-05-26 02:29\n"
    "- Aggregation summary (by cmdb_id, trace_name): top groups sorted by count. "
    "Example top group: cmdb_id=docker_001, trace_name=trace.from_os_022.remoteprocess.duration_mean, count=30, min≈0.2709, max≈0.5935, mean≈0.4280, p95≈0.5440, p50≈0.4422, p5≈0.3182. "
    "Many trace groups in the top results have count=30.\n\n"
    "Overall: the metric data is much larger (91k rows) and covers many KPI types and CMDBs; the trace data is smaller (~5.5k rows) with fewer CMDBs/traces. "
    "Top aggregated groups for metrics and traces show consistent per-group counts (metrics ~60–61, traces ~30) in the displayed top results."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(91076, 0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
51    redis_008
52    redis_009
53    redis_010
54    redis_011
55    redis_012
Length: 56, dtype: object, 0                   ACS
1                  AIOS
2                   AWS
3            Agent_ping
4           Asm_Free_Tb
             ...       
127        used_cpu_sys
128       used_cpu_user
129         used_memory
130    used_memory_peak
131     used_memory_rss
Length: 132, dtype: object, Timestamp('2020-05-26 01:30:00+0800', tz='Asia/Shanghai'), Timestamp('2020-05-26 02:30:00+0800', tz='Asia/Shanghai'),     cmdb_id                  kpi_name  count   min_value   max_value  mean_value         p95         p90         p50          p5
0   osb_001                   app.num     61  399.000000  635.000000  524.819672  574.000000  562.000000  531.000000  461.000000
1    os_017              Disk_io_util     61    0.000000    0.666667    0.019126    0.000000    0.000000    0.000000    0.000000
2   osb_001            app.succee_num     61  399.000000  635.000000  524.622951  574.000000  562.000000  531.000000  461.000000
3   osb_001           app.succee_rate     61    0.978100    1.000000    0.999641    1.000000    1.000000    1.000000    1.000000
4    os_015  Incoming_network_traffic     61    0.000000    0.006848    0.002162    0.003446    0.003416    0.002295    0.000000
..      ...                       ...    ...         ...         ...         ...         ...         ...         ...         ...
45   os_004                  ss_total     60  364.000000  414.000000  384.633333  408.000000  404.100000  383.000000  366.000000
46   os_019  Outgoing_network_traffic     60    0.000000    0.095352    0.056638    0.064561    0.063222    0.056849    0.049257
47   os_019           Memory_used_pct     60    8.550000    8.590000    8.573333    8.590000    8.590000    8.575000    8.550000
48   os_003             Disk_avgqu_sz     60    0.025244    0.076100    0.042402    0.066628    0.058372    0.044160    0.026029
49   os_003          Memory_available     60    8.633244    8.640976    8.638863    8.640569    8.640246    8.639273    8.636570

[50 rows x 10 columns], 5520, 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_007
6    docker_008
7        os_021
8        os_022
dtype: object, 0     trace.from_docker_001.remoteprocess.duration_mean
1      trace.from_docker_001.remoteprocess.duration_p95
2        trace.from_docker_001.remoteprocess.error_rate
3         trace.from_docker_001.remoteprocess.row_count
4     trace.from_docker_002.remoteprocess.duration_mean
                            ...                        
67          trace.to_docker_007.remoteprocess.row_count
68      trace.to_docker_008.remoteprocess.duration_mean
69       trace.to_docker_008.remoteprocess.duration_p95
70         trace.to_docker_008.remoteprocess.error_rate
71          trace.to_docker_008.remoteprocess.row_count
Length: 72, dtype: object, Timestamp('2020-05-26 02:00:00+0800', tz='Asia/Shanghai'), Timestamp('2020-05-26 02:29:00+0800', tz='Asia/Shanghai'),        cmdb_id                                       trace_name  count    min_value    max_value   mean_value          p95          p50           p5
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     30     0.270878     0.593480     0.427992     0.543952     0.442188     0.318236
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     30     0.510600     1.991100     1.360160     1.746850     1.402250     0.760500
2   docker_001       trace.from_os_022.remoteprocess.error_rate     30     0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count     30    95.000000   152.000000   128.166667   145.100000   129.000000   108.450000
4   docker_001                     trace.self.csf.duration_mean     30     0.036844     0.060028     0.047067     0.054789     0.046782     0.039538
..         ...                                              ...    ...          ...          ...          ...          ...          ...          ...
45  docker_002                    trace.self.local.duration_p95     30     0.078000     0.098000     0.091270     0.095000     0.093000     0.080102
46  docker_002                      trace.self.local.error_rate     30     0.000000     0.001553     0.000052     0.000000     0.000000     0.000000
47  docker_002                       trace.self.local.row_count     30  1484.000000  2100.000000  1820.466667  2062.200000  1813.000000  1610.000000
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean     30     0.026301     0.031068     0.028679     0.030771     0.028588     0.026735
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95     30     0.037400     0.062800     0.044552     0.060000     0.042850     0.037500

[50 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing metric_df loaded previously in the kernel.
# If metric_df not present, load it (defensive).
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Candidate lists
nodes = [f"os_{i:03d}" for i in range(1, 23)]           # os_001 .. os_022
pods  = [f"docker_{i:03d}" for i in range(1, 9)]         # docker_001 .. docker_008
svcs  = [f"db_{i:03d}" for i in range(1, 14)]           # db_001 .. db_013
candidates = set(nodes + pods + svcs)

# Filter metric_df to only candidate cmdb_ids
df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# If no candidate rows remain, return explicit empty list message
if df_cand.empty:
    anomaly_summary = pd.DataFrame()  # empty
    message = "No metric rows found for the provided candidate cmdb_id lists. Empty list."
    anomaly_summary, message
else:
    # 2) Compute global P95 and P5 thresholds per (cmdb_id, kpi_name) using full series (global)
    global_thresh = df_cand.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p5 =lambda x: x.quantile(0.05)
    ).reset_index()

    # 3) Filter metric rows to incident window May 26, 2020 02:00 to 02:30 (Asia/Shanghai) inclusive
    tz = pytz.timezone('Asia/Shanghai')
    start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
    end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
    df_win = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()

    # 4) For each (cmdb_id, kpi_name) compute required window stats and anomaly counts using global thresholds
    if df_win.empty:
        anomaly_summary = pd.DataFrame()  # empty
        message = "No metric rows within the incident window for candidate components. Empty list."
        anomaly_summary, message
    else:
        # merge thresholds into window rows
        df_win = df_win.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

        # mark high/low using global thresholds
        df_win['is_high'] = df_win['value'] >= df_win['global_p95']
        df_win['is_low']  = df_win['value'] <= df_win['global_p5']

        # group and summarize
        def summarize(g):
            count_window = int(len(g))
            count_high = int(g['is_high'].sum())
            count_low = int(g['is_low'].sum())
            earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high>0 else pd.NaT
            earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low>0 else pd.NaT
            max_in_win = g['value'].max()
            min_in_win = g['value'].min()
            return pd.Series({
                'count_window': count_window,
                'count_high': count_high,
                'count_low': count_low,
                'earliest_high_timestamp': earliest_high,
                'earliest_low_timestamp': earliest_low,
                'max_value_in_window': max_in_win,
                'min_value_in_window': min_in_win
            })

        win_summary = df_win.groupby(['cmdb_id','kpi_name']).apply(summarize).reset_index()

        # attach global thresholds
        summary = win_summary.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

        # format earliest timestamps as ISO strings (compact) or keep NaN for missing
        summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

        # Replace 'NaT' string with None for clarity
        summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].replace({pd.NaT: None})
        summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].replace({pd.NaT: None})

        # compute anomaly score and filter/sort
        summary['anomaly_count'] = summary['count_high'] + summary['count_low']
        summary = summary.sort_values(['anomaly_count','count_window'], ascending=[False, False])

        # 5) Keep only requested columns and top 20
        result_cols = [
            'cmdb_id','kpi_name','count_window','count_high','count_low',
            'earliest_high_timestamp','earliest_low_timestamp',
            'max_value_in_window','min_value_in_window','global_p95','global_p5'
        ]
        anomaly_summary = summary[result_cols].head(20).reset_index(drop=True)

        # If all anomaly counts zero, return explicit empty list message
        if anomaly_summary['count_high'].sum() + anomaly_summary['count_low'].sum() == 0:
            anomaly_summary = pd.DataFrame()
            message = "No anomalies found in the incident window (all count_high and count_low are zero). Empty list."
            anomaly_summary, message
        else:
            # compact display variable
            anomaly_summary

```
Out[2]:
```
summary = (
    "The last cell executed but returned None, so no anomaly table was emitted. "
    "This typically means the cell did not end by returning the result variables. "
    "To get the actual outcome, inspect the kernel variables 'anomaly_summary' and 'message', "
    "or re-run the analysis cell and ensure it ends with 'anomaly_summary' (or 'anomaly_summary, message'). "
    "If you want, I can re-run the preprocessing/anomaly cell and explicitly display the top results."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure types and timezone (UTC+8)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Candidate lists: os_001..os_022, docker_001..docker_008, db_001..db_013
nodes = [f"os_{i:03d}" for i in range(1, 23)]
pods  = [f"docker_{i:03d}" for i in range(1, 9)]
svcs  = [f"db_{i:03d}" for i in range(1, 14)]
candidates = set(nodes + pods + svcs)

df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 2) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (before time filtering)
if df_cand.empty:
    anomaly_summary = pd.DataFrame()
    message = 'no anomalies found in window'
    anomaly_summary, message
else:
    global_thresh = df_cand.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p5 =lambda x: x.quantile(0.05)
    ).reset_index()

    # 3) Filter to incident window May 26, 2020 02:00 to 02:30 (Asia/Shanghai) inclusive
    tz = pytz.timezone('Asia/Shanghai')
    start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
    end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
    df_win = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()

    if df_win.empty:
        anomaly_summary = pd.DataFrame()
        message = 'no anomalies found in window'
        anomaly_summary, message
    else:
        # 4) Merge thresholds and mark high/low
        df_win = df_win.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
        df_win['is_high'] = df_win['value'] >= df_win['global_p95']
        df_win['is_low']  = df_win['value'] <= df_win['global_p5']

        # Group and compute required aggregates
        def summarize(g):
            count_window = int(len(g))
            count_high = int(g['is_high'].sum())
            count_low = int(g['is_low'].sum())
            earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high>0 else pd.NaT
            earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low>0 else pd.NaT
            max_in_win = g['value'].max()
            min_in_win = g['value'].min()
            return pd.Series({
                'count_window': count_window,
                'count_high': count_high,
                'count_low': count_low,
                'earliest_high_timestamp': earliest_high,
                'earliest_low_timestamp': earliest_low,
                'max_value_in_window': max_in_win,
                'min_value_in_window': min_in_win
            })

        win_summary = df_win.groupby(['cmdb_id','kpi_name']).apply(summarize).reset_index()

        # attach global thresholds
        summary = win_summary.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

        # format earliest timestamps to compact strings (or None)
        summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
        summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)

        # anomaly count and filter/sort
        summary['anomaly_count'] = summary['count_high'] + summary['count_low']
        summary_pos = summary[summary['anomaly_count'] > 0].copy()

        if summary_pos.empty:
            anomaly_summary = pd.DataFrame()
            message = 'no anomalies found in window'
            anomaly_summary, message
        else:
            summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
            result_cols = [
                'cmdb_id','kpi_name','count_window','count_high','count_low',
                'earliest_high_timestamp','earliest_low_timestamp',
                'max_value_in_window','min_value_in_window','global_p95','global_p5'
            ]
            anomaly_summary = summary_pos[result_cols].head(20).reset_index(drop=True)
            message = ''
            anomaly_summary, message

```
Out[3]:
```
summary = (
    "The last cell executed but returned None (no visible output). This usually means the notebook cell "
    "did not end by returning the result variables. To get the detection outcome:\n"
    "- Inspect the kernel variables 'anomaly_summary' and 'message'.\n"
    "- If anomalies were found, 'anomaly_summary' will be a DataFrame (up to 20 rows) with these columns: "
    "cmdb_id, kpi_name, count_window, count_high, count_low, earliest_high_timestamp, earliest_low_timestamp, "
    "max_value_in_window, min_value_in_window, global_p95, global_p5.\n"
    "- If no anomalies were found, 'message' will be the string: 'no anomalies found in window'.\n\n"
    "If you want, I can re-run the analysis and explicitly return (anomaly_summary, message) so the table is printed."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse metric_df if present, otherwise load it
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure correct dtypes and timezone (UTC+8)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# If anomaly_summary/message already exist, display them compactly
try:
    anomaly_summary  # check existence
    # If message exists, include it; otherwise empty string
    try:
        message
    except NameError:
        message = ''
    # Compact view: top 20 rows (already expected to be compact)
    existing_anomaly = anomaly_summary.head(20) if isinstance(anomaly_summary, pd.DataFrame) else anomaly_summary
    existing_anomaly, message
except NameError:
    # Need to compute anomaly_summary and message
    # 1) Candidate lists
    nodes = [f"os_{i:03d}" for i in range(1, 23)]
    pods  = [f"docker_{i:03d}" for i in range(1, 9)]
    svcs  = [f"db_{i:03d}" for i in range(1, 14)]
    candidates = set(nodes + pods + svcs)

    # Restrict metric rows to candidates only
    df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

    if df_cand.empty:
        anomaly_summary = pd.DataFrame()
        message = 'no anomalies found in window'
        anomaly_summary, message
    else:
        # 2) Compute global thresholds (P95 and P5) per (cmdb_id, kpi_name) using full series
        global_thresh = df_cand.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
            global_p95=lambda x: x.quantile(0.95),
            global_p5 =lambda x: x.quantile(0.05)
        ).reset_index()

        # 3) Filter to incident window (Asia/Shanghai)
        tz = pytz.timezone('Asia/Shanghai')
        start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
        end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
        df_win = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()

        if df_win.empty:
            anomaly_summary = pd.DataFrame()
            message = 'no anomalies found in window'
            anomaly_summary, message
        else:
            # 4) Merge thresholds, mark highs/lows (comparison uses global thresholds)
            df_win = df_win.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
            df_win['is_high'] = df_win['value'] >= df_win['global_p95']
            df_win['is_low']  = df_win['value'] <= df_win['global_p5']

            # Aggregate per group for window
            def summarize(g):
                count_window = int(len(g))
                count_high = int(g['is_high'].sum())
                count_low = int(g['is_low'].sum())
                earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high>0 else pd.NaT
                earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low>0 else pd.NaT
                max_in_win = g['value'].max()
                min_in_win = g['value'].min()
                return pd.Series({
                    'count_window': count_window,
                    'count_high': count_high,
                    'count_low': count_low,
                    'earliest_high_timestamp': earliest_high,
                    'earliest_low_timestamp': earliest_low,
                    'max_value_in_window': max_in_win,
                    'min_value_in_window': min_in_win
                })

            win_summary = df_win.groupby(['cmdb_id','kpi_name']).apply(summarize).reset_index()

            # Attach global thresholds
            summary = win_summary.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

            # Format earliest timestamps as compact strings with timezone offset, replace NaT with None
            summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
            summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
            summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
            summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)

            # Compute anomaly_count and filter/sort
            summary['anomaly_count'] = summary['count_high'] + summary['count_low']
            summary_pos = summary[summary['anomaly_count'] > 0].copy()

            if summary_pos.empty:
                anomaly_summary = pd.DataFrame()
                message = 'no anomalies found in window'
                anomaly_summary, message
            else:
                summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
                result_cols = [
                    'cmdb_id','kpi_name','count_window','count_high','count_low',
                    'earliest_high_timestamp','earliest_low_timestamp',
                    'max_value_in_window','min_value_in_window','global_p95','global_p5'
                ]
                anomaly_summary = summary_pos[result_cols].head(20).reset_index(drop=True)
                message = ''
                anomaly_summary, message

```
Out[4]:
```
import pandas as pd
import pytz

# Helper to run the preprocessing & detection (same logic as before)
def compute_anomalies():
    # Load metric.csv if metric_df not present
    try:
        metric_df
    except NameError:
        df_all = pd.read_csv("metric.csv")
    else:
        df_all = metric_df.copy()
    # Ensure types and timezone
    df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
    df_all['ts_dt'] = pd.to_datetime(df_all['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
    # Candidate lists
    nodes = [f"os_{i:03d}" for i in range(1, 23)]
    pods  = [f"docker_{i:03d}" for i in range(1, 9)]
    svcs  = [f"db_{i:03d}" for i in range(1, 14)]
    candidates = set(nodes + pods + svcs)
    df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()
    if df_cand.empty:
        return pd.DataFrame(), 'no anomalies found in window'
    # Global thresholds per group (full series)
    global_thresh = df_cand.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p5 =lambda x: x.quantile(0.05)
    ).reset_index()
    # Filter to incident window (inclusive)
    tz = pytz.timezone('Asia/Shanghai')
    start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
    end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
    df_win = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()
    if df_win.empty:
        return pd.DataFrame(), 'no anomalies found in window'
    # Merge thresholds and mark highs/lows
    df_win = df_win.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
    df_win['is_high'] = df_win['value'] >= df_win['global_p95']
    df_win['is_low']  = df_win['value'] <= df_win['global_p5']
    # Aggregate per group within window
    def summarize(g):
        count_window = int(len(g))
        count_high = int(g['is_high'].sum())
        count_low = int(g['is_low'].sum())
        earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high>0 else pd.NaT
        earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low>0 else pd.NaT
        max_in_win = g['value'].max()
        min_in_win = g['value'].min()
        return pd.Series({
            'count_window': count_window,
            'count_high': count_high,
            'count_low': count_low,
            'earliest_high_timestamp': earliest_high,
            'earliest_low_timestamp': earliest_low,
            'max_value_in_window': max_in_win,
            'min_value_in_window': min_in_win
        })
    win_summary = df_win.groupby(['cmdb_id','kpi_name']).apply(summarize).reset_index()
    summary = win_summary.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
    # Format timestamps compactly
    summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
    summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)
    # Filter to only groups with anomalies (high or low) and sort
    summary['anomaly_count'] = summary['count_high'] + summary['count_low']
    summary_pos = summary[summary['anomaly_count'] > 0].copy()
    if summary_pos.empty:
        return pd.DataFrame(), 'no anomalies found in window'
    summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
    result_cols = [
        'cmdb_id','kpi_name','count_window','count_high','count_low',
        'earliest_high_timestamp','earliest_low_timestamp',
        'max_value_in_window','min_value_in_window','global_p95','global_p5'
    ]
    anomaly_summary_local = summary_pos[result_cols].head(20).reset_index(drop=True)
    return anomaly_summary_local, ''

# Main: check for existing kernel variables anomaly_summary and message
try:
    _ = anomaly_summary  # exists
    # prepare compact view
    if isinstance(anomaly_summary, pd.DataFrame) and not anomaly_summary.empty:
        # summarize: number of anomalous groups and top group's details
        n_groups = len(anomaly_summary)
        top = anomaly_summary.iloc[0]
        summary_str = (
            f"Anomalies already computed: {n_groups} group(s) with anomalies in window. "
            f"Top group: {top['cmdb_id']} | {top['kpi_name']} — count_window={top['count_window']}, "
            f"count_high={top['count_high']}, count_low={top['count_low']}."
        )
        display_df = anomaly_summary.head(20)
    else:
        # anomaly_summary exists but empty
        try:
            msg = message
        except NameError:
            msg = 'no anomalies found in window'
        summary_str = msg
        display_df = pd.DataFrame()
    # Return compact display and summary
    display_df, summary_str
except NameError:
    # Variables not present: compute them now
    anomaly_summary, message = compute_anomalies()
    if isinstance(anomaly_summary, pd.DataFrame) and not anomaly_summary.empty:
        n_groups = len(anomaly_summary)
        top = anomaly_summary.iloc[0]
        summary_str = (
            f"Anomalies found: {n_groups} group(s) with anomalies in window. "
            f"Top group: {top['cmdb_id']} | {top['kpi_name']} — count_window={top['count_window']}, "
            f"count_high={top['count_high']}, count_low={top['count_low']}."
        )
        display_df = anomaly_summary.head(20)
    else:
        summary_str = 'no anomalies found in window'
        display_df = pd.DataFrame()
    display_df, summary_str

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse metric_df if already in kernel, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric and timezone-aware datetime (Asia/Shanghai UTC+8)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'ts_dt' not in metric_df.columns:
    metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
else:
    # ensure tz and dtype consistency
    metric_df['ts_dt'] = pd.to_datetime(metric_df['ts_dt']).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Candidate lists
nodes = [f"os_{i:03d}" for i in range(1, 23)]
pods  = [f"docker_{i:03d}" for i in range(1, 9)]
svcs  = [f"db_{i:03d}" for i in range(1, 14)]
candidates = set(nodes + pods + svcs)

df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Prepare default outputs
anomaly_summary = pd.DataFrame()
message = 'no anomalies found in window'

# If there are candidate rows, proceed
if not df_cand.empty:
    # 2) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (before time filtering)
    global_thresh = df_cand.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p5 =lambda x: x.quantile(0.05)
    ).reset_index()

    # 3) Filter to incident window (inclusive) May 26, 2020 02:00:00 to 02:30:00 Asia/Shanghai
    tz = pytz.timezone('Asia/Shanghai')
    start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
    end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
    df_win = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()

    if not df_win.empty:
        # 4) Merge thresholds and mark high/low using global thresholds
        df_win = df_win.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
        df_win['is_high'] = df_win['value'] >= df_win['global_p95']
        df_win['is_low']  = df_win['value'] <= df_win['global_p5']

        # Aggregate per (cmdb_id, kpi_name) within the window
        def summarize(g):
            count_window = int(len(g))
            count_high = int(g['is_high'].sum())
            count_low = int(g['is_low'].sum())
            earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high > 0 else pd.NaT
            earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low > 0 else pd.NaT
            max_in_win = g['value'].max()
            min_in_win = g['value'].min()
            return pd.Series({
                'count_window': count_window,
                'count_high': count_high,
                'count_low': count_low,
                'earliest_high_timestamp': earliest_high,
                'earliest_low_timestamp': earliest_low,
                'max_value_in_window': max_in_win,
                'min_value_in_window': min_in_win
            })

        win_summary = df_win.groupby(['cmdb_id','kpi_name']).apply(summarize).reset_index()

        # Attach global thresholds
        summary = win_summary.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

        # Format earliest timestamps as compact strings (keep timezone offset); replace NaT with None
        summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
        summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)

        # 5) Keep only groups that have anomalies (count_high + count_low > 0)
        summary['anomaly_count'] = summary['count_high'] + summary['count_low']
        summary_pos = summary[summary['anomaly_count'] > 0].copy()

        if not summary_pos.empty:
            # Order by anomaly_count desc, then count_window desc and keep top 20
            summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
            result_cols = [
                'cmdb_id','kpi_name','count_window','count_high','count_low',
                'earliest_high_timestamp','earliest_low_timestamp',
                'max_value_in_window','min_value_in_window','global_p95','global_p5'
            ]
            anomaly_summary = summary_pos[result_cols].head(20).reset_index(drop=True)
            message = ''  # anomalies found -> empty message per spec
        else:
            # No anomalies in window; anomaly_summary remains empty and message set
            anomaly_summary = pd.DataFrame()
            message = 'no anomalies found in window'
    else:
        # No rows in incident window
        anomaly_summary = pd.DataFrame()
        message = 'no anomalies found in window'
else:
    # No candidate cmdb_ids found in metric.csv after filtering
    anomaly_summary = pd.DataFrame()
    message = 'no anomalies found in window'

# 7) Ensure variables are stored in kernel and return them (explicit final expression)
anomaly_summary, message

```
Out[5]:
```
summary = (
    "Anomaly detection result (metric.csv, window 2020-05-26 02:00–02:30 Asia/Shanghai):\n"
    "- Anomalies found: 20 groups (top 20 returned).\n"
    "- Top examples:\n"
    "  * os_010 | Disk_io_util — 31 points in window, count_high=31, count_low=31 (earliest 2020-05-26 02:00:00+0800)\n"
    "  * os_017 | Disk_io_util — 31 points, count_high=31, count_low=29\n"
    "  * db_001 | Hang, MEM_Total, On_Off_State, Row_Lock, SctRead_Per_Sec — each 30 points, count_high=30, count_low=30\n"
    "  * db_002, db_003, db_004 — multiple metrics with count_window=30 and both count_high and count_low = 30\n"
    "- Note: Many groups have count_high and count_low equal to the group window size. This occurs when global P95 and P5 equal the observed values (constant or identical-distribution series), so points satisfy both >=P95 and <=P5. This is a thresholding artifact rather than separate high/low deviations.\n"
    "If you want, I can (a) show the anomaly_summary DataFrame, (b) recompute using stricter anomaly rules, or (c.1) exclude cases where global_p95==global_p5 to avoid double-counting."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id         kpi_name  count_window  count_high  count_low   earliest_high_timestamp    earliest_low_timestamp  max_value_in_window  min_value_in_window  global_p95   global_p5
0   os_010     Disk_io_util            31          31         31  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
1   os_017     Disk_io_util            31          31         29  2020-05-26 02:00:00+0800  2020-05-26 02:02:00+0800             0.666667             0.000000    0.000000    0.000000
2   db_001             Hang            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
3   db_001        MEM_Total            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800           381.902264           381.902264  381.902264  381.902264
4   db_001     On_Off_State            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
5   db_001         Row_Lock            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
6   db_001  SctRead_Per_Sec            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
7   db_002             AIOS            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
8   db_002             Hang            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
9   db_002        MEM_Total            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800           381.902264           381.902264  381.902264  381.902264
10  db_002     On_Off_State            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
11  db_002         Row_Lock            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
12  db_002  SctRead_Per_Sec            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
13  db_003             Hang            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
14  db_003        MEM_Total            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800           381.902264           381.902264  381.902264  381.902264
15  db_003     On_Off_State            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
16  db_003         Row_Lock            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000
17  db_004             Hang            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
18  db_004     On_Off_State            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             1.000000             1.000000    1.000000    1.000000
19  db_004         Row_Lock            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800             0.000000             0.000000    0.000000    0.000000, '')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing anomaly_summary if present; otherwise try compute_anomalies if available, else recompute.
if 'anomaly_summary' in globals() and isinstance(anomaly_summary, pd.DataFrame):
    orig_anomaly = anomaly_summary.copy()
    # If message exists, keep it
    message = message if 'message' in globals() else ''
else:
    # try using compute_anomalies() if defined
    if 'compute_anomalies' in globals():
        orig_anomaly, message = compute_anomalies()
    else:
        # Recompute same way as before (compact)
        try:
            metric_df
        except NameError:
            metric_df = pd.read_csv("metric.csv")
        metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
        metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
        nodes = [f"os_{i:03d}" for i in range(1, 23)]
        pods  = [f"docker_{i:03d}" for i in range(1, 9)]
        svcs  = [f"db_{i:03d}" for i in range(1, 14)]
        candidates = set(nodes + pods + svcs)
        df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
        if df_cand.empty:
            orig_anomaly = pd.DataFrame()
            message = 'no anomalies found in window'
        else:
            # global thresholds
            global_thresh = df_cand.groupby(['cmdb_id','kpi_name'])['value'].agg(
                global_p95=lambda x: x.quantile(0.95),
                global_p5 =lambda x: x.quantile(0.05)
            ).reset_index()
            # window
            tz = pytz.timezone('Asia/Shanghai')
            start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
            end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
            df_win = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()
            if df_win.empty:
                orig_anomaly = pd.DataFrame()
                message = 'no anomalies found in window'
            else:
                df_win = df_win.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
                df_win['is_high'] = df_win['value'] >= df_win['global_p95']
                df_win['is_low']  = df_win['value'] <= df_win['global_p5']
                def summarize(g):
                    count_window = int(len(g))
                    count_high = int(g['is_high'].sum())
                    count_low = int(g['is_low'].sum())
                    earliest_high = g.loc[g['is_high'],'ts_dt'].min() if count_high>0 else pd.NaT
                    earliest_low  = g.loc[g['is_low'],'ts_dt'].min() if count_low>0 else pd.NaT
                    return pd.Series({
                        'count_window': count_window,
                        'count_high': count_high,
                        'count_low': count_low,
                        'earliest_high_timestamp': earliest_high,
                        'earliest_low_timestamp': earliest_low,
                        'max_value_in_window': g['value'].max(),
                        'min_value_in_window': g['value'].min()
                    })
                win_summary = df_win.groupby(['cmdb_id','kpi_name']).apply(summarize).reset_index()
                summary = win_summary.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
                summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
                summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
                summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
                summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)
                summary['anomaly_count'] = summary['count_high'] + summary['count_low']
                summary_pos = summary[summary['anomaly_count'] > 0].copy()
                if summary_pos.empty:
                    orig_anomaly = pd.DataFrame()
                    message = 'no anomalies found in window'
                else:
                    summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
                    cols = [
                        'cmdb_id','kpi_name','count_window','count_high','count_low',
                        'earliest_high_timestamp','earliest_low_timestamp',
                        'max_value_in_window','min_value_in_window','global_p95','global_p5'
                    ]
                    orig_anomaly = summary_pos[cols].head(20).reset_index(drop=True)
                    message = ''

# Now orig_anomaly holds the original anomalous groups DataFrame (or empty)
n_total = int(len(orig_anomaly))  # original number of anomalous groups (top set)
if n_total == 0:
    anomaly_filtered = pd.DataFrame()
    n_filtered_out = 0
    message = 'no valid anomalies after filtering constants'  # per spec when none remain
else:
    # Filter out groups where global_p95 == global_p5 (use np.isclose for numeric stability)
    mask_diff = ~np.isclose(orig_anomaly['global_p95'].astype(float), orig_anomaly['global_p5'].astype(float))
    anomaly_filtered = orig_anomaly[mask_diff].copy()
    n_filtered_out = int(n_total - len(anomaly_filtered))
    if anomaly_filtered.empty:
        message = 'no valid anomalies after filtering constants'
    else:
        # Order by (count_high + count_low) desc
        anomaly_filtered['anomaly_count'] = anomaly_filtered['count_high'] + anomaly_filtered['count_low']
        anomaly_filtered = anomaly_filtered.sort_values(['anomaly_count','count_window'], ascending=[False, False])
        anomaly_filtered = anomaly_filtered.drop(columns=['anomaly_count']).reset_index(drop=True)
        message = ''

# Return compact results
n_total, n_filtered_out, anomaly_filtered.head(20), message

```
Out[6]:
```
Summary of results:

- n_total: 20 anomalous groups were originally identified.
- n_filtered_out: All 20 groups were filtered out because global_p95 == global_p5 (constant-series artifact).
- anomaly_filtered: empty DataFrame — no valid anomalies remain after filtering.
- message: "no valid anomalies after filtering constants"

Interpretation: every flagged group had identical P95 and P5 (constant or nearly-constant series), so they were removed as thresholding artifacts. If you want, I can (a) show the original groups before filtering, (b) apply a different anomaly rule (e.g., require global_p95 > global_p5 and value deviations beyond a margin), or (c) inspect raw series for any true deviations. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(20, 20, Empty DataFrame
Columns: [cmdb_id, kpi_name, count_window, count_high, count_low, earliest_high_timestamp, earliest_low_timestamp, max_value_in_window, min_value_in_window, global_p95, global_p5]
Index: [], 'no valid anomalies after filtering constants')```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric and timezone-aware datetime (Asia/Shanghai UTC+8)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
if 'ts_dt' not in trace_df.columns:
    trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
else:
    trace_df['ts_dt'] = pd.to_datetime(trace_df['ts_dt']).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Compute global thresholds (P95, P5) per (cmdb_id, trace_name) using full series
global_thresh_trace = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p5 =lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter to incident window (inclusive) May 26, 2020 02:00:00 to 02:30:00 Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
df_win = trace_df[(trace_df['ts_dt'] >= start) & (trace_df['ts_dt'] <= end)].copy()

# Prepare defaults
trace_anomalies = pd.DataFrame()
message = 'no trace anomalies found in window'

if not df_win.empty:
    # Merge global thresholds
    df_win = df_win.merge(global_thresh_trace, on=['cmdb_id','trace_name'], how='left')

    # Mark highs and lows using global thresholds (computed on full series)
    df_win['is_high'] = df_win['value'] >= df_win['global_p95']
    df_win['is_low']  = df_win['value'] <= df_win['global_p5']

    # Aggregate per (cmdb_id, trace_name) within window
    def summarize(g):
        count_window = int(len(g))
        count_high = int(g['is_high'].sum())
        count_low = int(g['is_low'].sum())
        earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high>0 else pd.NaT
        earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low>0 else pd.NaT
        return pd.Series({
            'count_window': count_window,
            'count_high': count_high,
            'count_low': count_low,
            'earliest_high_timestamp': earliest_high,
            'earliest_low_timestamp': earliest_low,
            'max_value_in_window': g['value'].max(),
            'min_value_in_window': g['value'].min()
        })

    win_summary = df_win.groupby(['cmdb_id','trace_name']).apply(summarize).reset_index()

    # Attach global thresholds
    summary = win_summary.merge(global_thresh_trace, on=['cmdb_id','trace_name'], how='left')

    # Format earliest timestamps as compact strings with timezone; replace NaT with None
    summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
    summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)

    # Keep only groups with anomalies (count_high + count_low > 0)
    summary['anomaly_count'] = summary['count_high'] + summary['count_low']
    summary_pos = summary[summary['anomaly_count'] > 0].copy()

    if not summary_pos.empty:
        # Order by anomaly_count desc, then count_window desc, limit to 20
        summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
        cols = [
            'cmdb_id','trace_name','count_window','count_high','count_low',
            'earliest_high_timestamp','earliest_low_timestamp',
            'max_value_in_window','min_value_in_window','global_p95','global_p5'
        ]
        trace_anomalies = summary_pos[cols].head(20).reset_index(drop=True)
        message = ''  # anomalies present -> empty message

# Return the requested aggregates compactly
trace_anomalies, message

```
Out[7]:
```
summary = (
    "Trace anomaly detection (window 2020-05-26 02:00–02:30 Asia/Shanghai):\n"
    "- Found 20 trace groups flagged as anomalous (top 20 returned).\n"
    "- Examples: docker_001 trace.from_os_022.remoteprocess.error_rate, docker_001 trace.self.csf.error_rate, docker_002 trace.self.flyremote.error_rate, etc.\n"
    "- Observations: for these groups count_window=30, count_high=30, count_low=30, max/min in window = 0.0, and global_p95 == global_p5 == 0.0.\n"
    "- Interpretation: these are constant (all-zero) series so every point meets both >=P95 and <=P5; this is a thresholding artifact rather than true high/low deviations.\n"
    "- Recommendation: filter out groups where global_p95 == global_p5 (constant-series) or require global_p95 > global_p5/introduce a margin before treating points as anomalies. I can apply that filter and return the cleaned list if you want."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                      trace_name  count_window  count_high  count_low   earliest_high_timestamp    earliest_low_timestamp  max_value_in_window  min_value_in_window  global_p95  global_p5
0   docker_001      trace.from_os_022.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
1   docker_001                       trace.self.csf.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
2   docker_001                 trace.self.flyremote.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
3   docker_001    trace.to_docker_007.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
4   docker_001    trace.to_docker_008.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
5   docker_002      trace.from_os_022.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
6   docker_002                       trace.self.csf.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
7   docker_002                 trace.self.flyremote.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
8   docker_002    trace.to_docker_007.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
9   docker_002    trace.to_docker_008.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
10  docker_003      trace.from_os_021.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
11  docker_003                       trace.self.csf.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
12  docker_003                 trace.self.flyremote.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
13  docker_003    trace.to_docker_005.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
14  docker_004      trace.from_os_021.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
15  docker_004                       trace.self.csf.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
16  docker_004                 trace.self.flyremote.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
17  docker_004    trace.to_docker_005.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
18  docker_005  trace.from_docker_003.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0
19  docker_005  trace.from_docker_004.remoteprocess.error_rate            30          30         30  2020-05-26 02:00:00+0800  2020-05-26 02:00:00+0800                  0.0                  0.0         0.0        0.0, '')```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric and timezone-aware datetime (Asia/Shanghai UTC+8)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series
global_thresh_trace = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p5 =lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter out constant-series groups where global_p95 == global_p5
non_const = global_thresh_trace[~np.isclose(global_thresh_trace['global_p95'].astype(float),
                                           global_thresh_trace['global_p5'].astype(float))].copy()

# Prepare default outputs
trace_candidates = pd.DataFrame()
message = 'no non-constant trace groups found'

# If any non-constant groups remain, proceed
if not non_const.empty:
    # 3) Filter trace rows to the incident window (inclusive)
    tz = pytz.timezone('Asia/Shanghai')
    start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
    end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
    df_win = trace_df[(trace_df['ts_dt'] >= start) & (trace_df['ts_dt'] <= end)].copy()

    if not df_win.empty:
        # Keep only rows belonging to non-constant groups by merging thresholds
        df_win = df_win.merge(non_const, on=['cmdb_id','trace_name'], how='inner')

        if not df_win.empty:
            # 4) Mark highs/lows using global thresholds (from full series)
            df_win['is_high'] = df_win['value'] >= df_win['global_p95']
            df_win['is_low']  = df_win['value'] <= df_win['global_p5']

            # Aggregate per (cmdb_id, trace_name)
            def summarize(g):
                count_window = int(len(g))
                count_high = int(g['is_high'].sum())
                count_low = int(g['is_low'].sum())
                earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high > 0 else pd.NaT
                earliest_low  = g.loc[g['is_low'],  'ts_dt'].min() if count_low > 0 else pd.NaT
                return pd.Series({
                    'count_window': count_window,
                    'count_high': count_high,
                    'count_low': count_low,
                    'earliest_high_timestamp': earliest_high,
                    'earliest_low_timestamp': earliest_low,
                    'max_value_in_window': g['value'].max(),
                    'min_value_in_window': g['value'].min()
                })

            win_summary = df_win.groupby(['cmdb_id','trace_name']).apply(summarize).reset_index()
            summary = win_summary.merge(non_const, on=['cmdb_id','trace_name'], how='left')

            # Format earliest timestamps to compact strings and replace NaT with None
            summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
            summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
            summary['earliest_high_timestamp'] = summary['earliest_high_timestamp'].where(summary['earliest_high_timestamp'].notna(), None)
            summary['earliest_low_timestamp']  = summary['earliest_low_timestamp'].where(summary['earliest_low_timestamp'].notna(), None)

            # Keep only groups with any anomalies (count_high + count_low > 0)
            summary['anomaly_count'] = summary['count_high'] + summary['count_low']
            summary_pos = summary[summary['anomaly_count'] > 0].copy()

            if not summary_pos.empty:
                # 5) Order by anomaly_count desc and limit to top 20
                summary_pos = summary_pos.sort_values(['anomaly_count','count_window'], ascending=[False, False])
                cols = [
                    'cmdb_id','trace_name','count_window','count_high','count_low',
                    'earliest_high_timestamp','earliest_low_timestamp',
                    'max_value_in_window','min_value_in_window','global_p95','global_p5'
                ]
                trace_candidates = summary_pos[cols].head(20).reset_index(drop=True)
                message = ''  # found candidates

# Return only the requested aggregates
trace_candidates, message

```
Out[8]:
```
Summary of trace analysis (window 2020-05-26 02:00–02:30 Asia/Shanghai):

- Non-constant trace groups found: 20 candidate groups (top 20 returned).
- All candidates have 30 points in the window (count_window = 30). They include duration_p95, local/different span durations, and row_count metrics across docker_001..docker_008.
- Example top rows:
  - docker_001 | trace.self.jdbc.duration_p95 — count_window=30, count_high=3, count_low=27, earliest_high=2020-05-26 02:11:00+0800, earliest_low=2020-05-26 02:00:00+0800, max=0.0040, min=0.0030, global_p95=0.00400, global_p5=0.00300.
  - docker_007 | trace.self.local.duration_p95 — count_window=30, count_high=12, count_low=18, earliest_high=2020-05-26 02:03:00+0800, earliest_low=2020-05-26 02:00:00+0800, max=0.0120, min=0.0100, global_p95=0.01100, global_p5=0.01000.
  - docker_004 | trace.self.jdbc.duration_p95 — count_window=30, count_high=2, count_low=27, earliest_high=2020-05-26 02:11:00+0800, global_p95≈0.00565, global_p5=0.00300.
  - Several candidates are row_count metrics (docker_002) with global_p95 > global_p5 and some points above/below thresholds.
- Interpretation: These are non-constant traces where some timestamps exceed the global P95 (count_high > 0) and/or fall below the global P5. The counts vary by trace: some have few highs (e.g., 2–5), some have more (e.g., 12).
- Next steps you might want: inspect the full time series for high-count traces (e.g., docker_007 trace.self.local.duration_p95) to understand root cause, or correlate these trace anomalies with metric anomalies on the same nodes/pods.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                        trace_name  count_window  count_high  count_low   earliest_high_timestamp    earliest_low_timestamp  max_value_in_window  min_value_in_window  global_p95    global_p5
0   docker_001                      trace.self.jdbc.duration_p95            30           3         27  2020-05-26 02:11:00+0800  2020-05-26 02:00:00+0800               0.0040              0.00300     0.00400     0.003000
1   docker_007                     trace.self.local.duration_p95            30          12         18  2020-05-26 02:03:00+0800  2020-05-26 02:00:00+0800               0.0120              0.01000     0.01100     0.010000
2   docker_004                      trace.self.jdbc.duration_p95            30           2         27  2020-05-26 02:11:00+0800  2020-05-26 02:00:00+0800               0.0100              0.00300     0.00565     0.003000
3   docker_003                      trace.self.jdbc.duration_p95            30           3         23  2020-05-26 02:04:00+0800  2020-05-26 02:00:00+0800               0.0180              0.00300     0.00700     0.003000
4   docker_002                      trace.self.jdbc.duration_p95            30           2         21  2020-05-26 02:04:00+0800  2020-05-26 02:01:00+0800               0.0140              0.00300     0.00510     0.003000
5   docker_008                     trace.self.local.duration_p95            30           2          9  2020-05-26 02:02:00+0800  2020-05-26 02:00:00+0800               0.0110              0.00900     0.01082     0.009000
6   docker_005                     trace.self.local.duration_p95            30           2          7  2020-05-26 02:09:00+0800  2020-05-26 02:02:00+0800               0.0160              0.01300     0.01555     0.013000
7   docker_001                 trace.self.flyremote.duration_p95            30           2          6  2020-05-26 02:10:00+0800  2020-05-26 02:05:00+0800               0.0862              0.00600     0.05936     0.006000
8   docker_002                     trace.self.local.duration_p95            30           5          2  2020-05-26 02:11:00+0800  2020-05-26 02:01:00+0800               0.0980              0.07800     0.09500     0.080102
9   docker_003                       trace.self.csf.duration_p95            30           5          2  2020-05-26 02:11:00+0800  2020-05-26 02:01:00+0800               0.1990              0.09900     0.19500     0.099900
10  docker_002    trace.to_docker_007.remoteprocess.duration_p95            30           3          3  2020-05-26 02:19:00+0800  2020-05-26 02:09:00+0800               0.0628              0.03740     0.06000     0.037500
11  docker_007  trace.from_docker_002.remoteprocess.duration_p95            30           3          3  2020-05-26 02:19:00+0800  2020-05-26 02:09:00+0800               0.0628              0.03740     0.06000     0.037500
12  docker_002         trace.from_os_022.remoteprocess.row_count            30           2          3  2020-05-26 02:11:00+0800  2020-05-26 02:22:00+0800             150.0000            106.00000   147.30000   115.000000
13  docker_002                          trace.self.csf.row_count            30           2          3  2020-05-26 02:11:00+0800  2020-05-26 02:22:00+0800             600.0000            424.00000   589.20000   460.000000
14  docker_002                    trace.self.flyremote.row_count            30           2          3  2020-05-26 02:11:00+0800  2020-05-26 02:22:00+0800             150.0000            106.00000   147.30000   115.000000
15  docker_002                         trace.self.jdbc.row_count            30           2          3  2020-05-26 02:11:00+0800  2020-05-26 02:22:00+0800            2405.0000           1700.00000  2359.60000  1843.000000
16  docker_002                        trace.self.local.row_count            30           2          3  2020-05-26 02:11:00+0800  2020-05-26 02:22:00+0800            2100.0000           1484.00000  2062.20000  1610.000000
17  docker_003                     trace.self.local.duration_p95            30           2          3  2020-05-26 02:22:00+0800  2020-05-26 02:01:00+0800               0.0980              0.07000     0.09710     0.079000
18  docker_004                     trace.self.local.duration_p95            30           3          2  2020-05-26 02:13:00+0800  2020-05-26 02:06:00+0800               0.0970              0.03200     0.09600     0.062537
19  docker_004    trace.to_docker_005.remoteprocess.duration_p95            30           2          3  2020-05-26 02:00:00+0800  2020-05-26 02:02:00+0800               0.0703              0.04265     0.06155     0.045000, '')```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import re

# Reuse existing trace_candidates if present; otherwise try other cached vars or recompute
if 'trace_candidates' in globals() and isinstance(trace_candidates, pd.DataFrame):
    df_candidates = trace_candidates.copy()
else:
    # try trace_anomalies (older name) if exists
    if 'trace_anomalies' in globals() and isinstance(trace_anomalies, pd.DataFrame):
        df_candidates = trace_anomalies.copy()
    else:
        # Recompute trace_candidates following prior logic
        try:
            trace_df
        except NameError:
            trace_df = pd.read_csv("trace.csv")
        trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
        trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
        # global thresholds
        global_thresh = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(
            global_p95=lambda x: x.quantile(0.95),
            global_p5 =lambda x: x.quantile(0.05)
        ).reset_index()
        # filter out constant groups
        non_const = global_thresh[~(global_thresh['global_p95'].astype(float).eq(global_thresh['global_p5'].astype(float)))].copy()
        # window
        tz = pytz.timezone('Asia/Shanghai')
        start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
        end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
        df_win = trace_df[(trace_df['ts_dt'] >= start) & (trace_df['ts_dt'] <= end)].copy()
        if df_win.empty or non_const.empty:
            df_candidates = pd.DataFrame()
        else:
            df_win = df_win.merge(non_const, on=['cmdb_id','trace_name'], how='inner')
            df_win['is_high'] = df_win['value'] >= df_win['global_p95']
            df_win['is_low']  = df_win['value'] <= df_win['global_p5']
            def summarize(g):
                count_window = int(len(g))
                count_high = int(g['is_high'].sum())
                count_low = int(g['is_low'].sum())
                earliest_high = g.loc[g['is_high'],'ts_dt'].min() if count_high>0 else pd.NaT
                earliest_low  = g.loc[g['is_low'],'ts_dt'].min() if count_low>0 else pd.NaT
                return pd.Series({
                    'count_window': count_window,
                    'count_high': count_high,
                    'count_low': count_low,
                    'earliest_high_timestamp': earliest_high,
                    'earliest_low_timestamp': earliest_low,
                    'max_value_in_window': g['value'].max(),
                    'min_value_in_window': g['value'].min()
                })
            win_summary = df_win.groupby(['cmdb_id','trace_name']).apply(summarize).reset_index()
            summary = win_summary.merge(non_const, on=['cmdb_id','trace_name'], how='left')
            # format timestamps as strings but keep original for sorting via to_datetime
            summary['earliest_high_timestamp_str'] = summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
            summary['earliest_high_timestamp_str'] = summary['earliest_high_timestamp_str'].where(summary['earliest_high_timestamp_str'].notna(), None)
            cols = [
                'cmdb_id','trace_name','count_window','count_high','count_low',
                'earliest_high_timestamp_str','max_value_in_window','min_value_in_window','global_p95','global_p5'
            ]
            df_candidates = summary.rename(columns={'earliest_high_timestamp_str':'earliest_high_timestamp'})[cols].copy()

# Now filter to groups with count_high > 0
if df_candidates.empty:
    trace_earliest_highs = pd.DataFrame()
else:
    df_pos = df_candidates[df_candidates['count_high'].astype(int) > 0].copy()
    if df_pos.empty:
        trace_earliest_highs = pd.DataFrame()
    else:
        # Ensure earliest_high_timestamp is parseable for sorting; convert to datetime with tz
        df_pos['earliest_high_dt'] = pd.to_datetime(df_pos['earliest_high_timestamp'], format='%Y-%m-%d %H:%M:%S%z', errors='coerce')
        # parse peer from trace_name
        def extract_peer(name):
            m_to = re.search(r'\bto_([A-Za-z0-9_]+)', name)
            if m_to:
                return m_to.group(1)
            m_from = re.search(r'\bfrom_([A-Za-z0-9_]+)', name)
            if m_from:
                return m_from.group(1)
            return 'self'
        df_pos['peer'] = df_pos['trace_name'].apply(extract_peer)
        # select requested columns and sort by earliest_high_timestamp ascending
        df_pos = df_pos.sort_values('earliest_high_dt', ascending=True)
        trace_earliest_highs = df_pos[[
            'cmdb_id','trace_name','peer','count_window','count_high','earliest_high_timestamp','max_value_in_window','global_p95'
        ]].head(20).reset_index(drop=True)

# Final output variable
trace_earliest_highs

```
Out[9]:
```
Summary of trace earliest-highs (trace_earliest_highs, up to 20 rows):

- Returned 20 candidate trace groups sorted by earliest_high_timestamp (ascending). Columns: cmdb_id, trace_name, peer, count_window, count_high, earliest_high_timestamp, max_value_in_window, global_p95.

- Earliest high:
  - 2020-05-26 02:00:00+0800 — docker_004 | trace.to_docker_005.remoteprocess.duration_p95 (peer: docker_005), count_window=30, count_high=2, max=0.0703, global_p95=0.06155.

- Highest number of high points:
  - docker_007 | trace.self.local.duration_p95 (peer: self), earliest_high=2020-05-26 02:03:00+0800, count_high=12 (count_window=30), max=0.0120, global_p95=0.01100.

- Other notable items (examples):
  - docker_003 | trace.self.jdbc.duration_p95 — earliest 02:04, count_high=3, max=0.0180, global_p95=0.00700.
  - docker_002 | trace.self.local.duration_p95 — earliest 02:11, count_high=5, max=0.0980, global_p95=0.09500.
  - Several row_count traces on docker_002 show small numbers of highs (earliest 02:11) with global_p95 > global_p5.

- Peer parsing: many traces are 'self' (internal), several involve peers like docker_005, docker_007, os_022, docker_002.

Interpretation / next steps (suggested):
- Prioritize investigation of docker_007 trace.self.local.duration_p95 (most highs) and early-peer anomalies (e.g., docker_004 → docker_005 at 02:00).
- Inspect full time series and related metrics on the involved pods/nodes to determine root cause (network, CPU, or downstream service).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                        trace_name        peer  count_window  count_high   earliest_high_timestamp  max_value_in_window  global_p95
0   docker_004    trace.to_docker_005.remoteprocess.duration_p95  docker_005            30           2  2020-05-26 02:00:00+0800               0.0703     0.06155
1   docker_008                     trace.self.local.duration_p95        self            30           2  2020-05-26 02:02:00+0800               0.0110     0.01082
2   docker_007                     trace.self.local.duration_p95        self            30          12  2020-05-26 02:03:00+0800               0.0120     0.01100
3   docker_003                      trace.self.jdbc.duration_p95        self            30           3  2020-05-26 02:04:00+0800               0.0180     0.00700
4   docker_002                      trace.self.jdbc.duration_p95        self            30           2  2020-05-26 02:04:00+0800               0.0140     0.00510
5   docker_005                     trace.self.local.duration_p95        self            30           2  2020-05-26 02:09:00+0800               0.0160     0.01555
6   docker_001                 trace.self.flyremote.duration_p95        self            30           2  2020-05-26 02:10:00+0800               0.0862     0.05936
7   docker_002                        trace.self.local.row_count        self            30           2  2020-05-26 02:11:00+0800            2100.0000  2062.20000
8   docker_002                         trace.self.jdbc.row_count        self            30           2  2020-05-26 02:11:00+0800            2405.0000  2359.60000
9   docker_002                    trace.self.flyremote.row_count        self            30           2  2020-05-26 02:11:00+0800             150.0000   147.30000
10  docker_002                          trace.self.csf.row_count        self            30           2  2020-05-26 02:11:00+0800             600.0000   589.20000
11  docker_002         trace.from_os_022.remoteprocess.row_count      os_022            30           2  2020-05-26 02:11:00+0800             150.0000   147.30000
12  docker_001                      trace.self.jdbc.duration_p95        self            30           3  2020-05-26 02:11:00+0800               0.0040     0.00400
13  docker_002                     trace.self.local.duration_p95        self            30           5  2020-05-26 02:11:00+0800               0.0980     0.09500
14  docker_004                      trace.self.jdbc.duration_p95        self            30           2  2020-05-26 02:11:00+0800               0.0100     0.00565
15  docker_003                       trace.self.csf.duration_p95        self            30           5  2020-05-26 02:11:00+0800               0.1990     0.19500
16  docker_004                     trace.self.local.duration_p95        self            30           3  2020-05-26 02:13:00+0800               0.0970     0.09600
17  docker_002    trace.to_docker_007.remoteprocess.duration_p95  docker_007            30           3  2020-05-26 02:19:00+0800               0.0628     0.06000
18  docker_007  trace.from_docker_002.remoteprocess.duration_p95  docker_002            30           3  2020-05-26 02:19:00+0800               0.0628     0.06000
19  docker_003                     trace.self.local.duration_p95        self            30           2  2020-05-26 02:22:00+0800               0.0980     0.09710```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric and timezone-aware datetime (Asia/Shanghai UTC+8)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Compute global P95 per (cmdb_id, trace_name) using full series (if not already computed)
global_thresh_trace = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p5 =lambda x: x.quantile(0.05)
).reset_index()

# Selected groups
selected = [
    ("docker_004", "trace.to_docker_005.remoteprocess.duration_p95"),
    ("docker_008", "trace.self.local.duration_p95"),
    ("docker_007", "trace.self.local.duration_p95"),
    ("docker_003", "trace.self.jdbc.duration_p95"),
    ("docker_002", "trace.self.jdbc.duration_p95"),
]

# Incident window
tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)

rows = []
no_high_groups = []

for cmdb_id, trace_name in selected:
    # get group's global thresholds
    th = global_thresh_trace[
        (global_thresh_trace['cmdb_id'] == cmdb_id) &
        (global_thresh_trace['trace_name'] == trace_name)
    ]
    if th.empty:
        global_p95 = None
        global_p5 = None
    else:
        global_p95 = float(th['global_p95'].iloc[0])
        global_p5  = float(th['global_p5'].iloc[0])

    # filter rows in window for this group
    g = trace_df[
        (trace_df['cmdb_id'] == cmdb_id) &
        (trace_df['trace_name'] == trace_name) &
        (trace_df['ts_dt'] >= start) & (trace_df['ts_dt'] <= end)
    ].copy().sort_values('ts_dt')

    if g.empty or global_p95 is None:
        count_high = 0
        earliest_high = None
        high_ts_list = []
        longest_len = 0
        longest_start = None
    else:
        # mark highs using global_p95
        g['is_high'] = g['value'] >= global_p95

        # count_high and earliest_high_timestamp
        count_high = int(g['is_high'].sum())
        if count_high > 0:
            earliest_high_dt = g.loc[g['is_high'], 'ts_dt'].min()
            earliest_high = earliest_high_dt.strftime('%Y-%m-%d %H:%M:%S%z')
            # list of timestamps where value >= global_p95, limit 10
            high_ts = g.loc[g['is_high'], 'ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z').tolist()[:10]
            high_ts_list = high_ts
        else:
            earliest_high = None
            high_ts_list = []

        # 2) Detect consecutive-high segments (run-length on the ordered rows)
        # Create run ids
        g['high_shift'] = g['is_high'].shift(fill_value=False)
        g['run_change'] = g['is_high'] != g['high_shift']
        g['run_id'] = g['run_change'].cumsum()
        # Group by run_id where is_high True to get lengths and starts
        high_runs = g[g['is_high']].groupby('run_id').agg(
            run_len=('ts_dt','count'),
            run_start=('ts_dt','min')
        ).reset_index()
        if not high_runs.empty:
            # longest run
            longest = high_runs.sort_values(['run_len','run_start'], ascending=[False, True]).iloc[0]
            longest_len = int(longest['run_len'])
            longest_start = longest['run_start'].strftime('%Y-%m-%d %H:%M:%S%z')
        else:
            longest_len = 0
            longest_start = None

    # parse peer field
    peer = 'self'
    import re
    m_to = re.search(r'\bto_([A-Za-z0-9_]+)', trace_name)
    m_from = re.search(r'\bfrom_([A-Za-z0-9_]+)', trace_name)
    if m_to:
        peer = m_to.group(1)
    elif m_from:
        peer = m_from.group(1)

    rows.append({
        'cmdb_id': cmdb_id,
        'trace_name': trace_name,
        'peer': peer,
        'count_high': count_high,
        'earliest_high_timestamp': earliest_high,
        'longest_segment_len_minutes': longest_len,
        'longest_segment_start': longest_start,
        'high_timestamps_list': high_ts_list,
        'max_value_in_window': (g['value'].max() if not g.empty else None),
        'global_p95': global_p95
    })

    if count_high == 0:
        no_high_groups.append(f"{cmdb_id} | {trace_name}")

# Build DataFrame (limit to selected five groups only, preserve order)
trace_high_segments = pd.DataFrame(rows, columns=[
    'cmdb_id','trace_name','peer','count_high','earliest_high_timestamp',
    'longest_segment_len_minutes','longest_segment_start','high_timestamps_list','max_value_in_window','global_p95'
])

# Prepare message if any group has no highs
if no_high_groups:
    message = "groups with no highs: " + "; ".join(no_high_groups)
else:
    message = ''

# Return compact aggregates
trace_high_segments, message

```
Out[10]:
```
Summary of results for the five selected trace groups (window 2020-05-26 02:00–02:30 Asia/Shanghai):

- docker_004 | trace.to_docker_005.remoteprocess.duration_p95 (peer: docker_005)
  - count_high: 2
  - earliest_high_timestamp: 2020-05-26 02:00:00+0800
  - longest consecutive-high segment: 1 minute, starting 2020-05-26 02:00:00+0800
  - sample high timestamps (up to 10): 02:00:00+0800, 02:17:00+0800, ...
  - max_value_in_window: 0.0703, global_p95: 0.06155

- docker_008 | trace.self.local.duration_p95 (peer: self)
  - count_high: 2
  - earliest_high_timestamp: 2020-05-26 02:02:00+0800
  - longest consecutive-high segment: 1 minute, starting 2020-05-26 02:02:00+0800
  - sample high timestamps: 02:02:00+0800, 02:27:00+0800, ...
  - max_value_in_window: 0.0110, global_p95: 0.01082

- docker_007 | trace.self.local.duration_p95 (peer: self)
  - count_high: 12
  - earliest_high_timestamp: 2020-05-26 02:03:00+0800
  - longest consecutive-high segment: 3 minutes, starting 2020-05-26 02:20:00+0800
  - sample high timestamps: 02:03:00+0800, 02:04:00+0800, ... (up to 10)
  - max_value_in_window: 0.0120, global_p95: 0.01100

- docker_003 | trace.self.jdbc.duration_p95 (peer: self)
  - count_high: 3
  - earliest_high_timestamp: 2020-05-26 02:04:00+0800
  - longest consecutive-high segment: 1 minute, starting 2020-05-26 02:04:00+0800
  - sample high timestamps: 02:04:00+0800, 02:11:00+0800, ...
  - max_value_in_window: 0.0180, global_p95: 0.00700

- docker_002 | trace.self.jdbc.duration_p95 (peer: self)
  - count_high: 2
  - earliest_high_timestamp: 2020-05-26 02:04:00+0800
  - longest consecutive-high segment: 1 minute, starting 2020-05-26 02:04:00+0800
  - sample high timestamps: 02:04:00+0800, 02:11:00+0800, ...
  - max_value_in_window: 0.0140, global_p95: 0.00510

Message: (no groups with zero highs) — all five groups had at least one high. Priority for investigation: docker_007 (most highs and a 3-minute consecutive-high segment).

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                      trace_name        peer  count_high   earliest_high_timestamp  longest_segment_len_minutes     longest_segment_start                               high_timestamps_list  max_value_in_window  global_p95
0  docker_004  trace.to_docker_005.remoteprocess.duration_p95  docker_005           2  2020-05-26 02:00:00+0800                            1  2020-05-26 02:00:00+0800  [2020-05-26 02:00:00+0800, 2020-05-26 02:17:00...               0.0703     0.06155
1  docker_008                   trace.self.local.duration_p95        self           2  2020-05-26 02:02:00+0800                            1  2020-05-26 02:02:00+0800  [2020-05-26 02:02:00+0800, 2020-05-26 02:27:00...               0.0110     0.01082
2  docker_007                   trace.self.local.duration_p95        self          12  2020-05-26 02:03:00+0800                            3  2020-05-26 02:20:00+0800  [2020-05-26 02:03:00+0800, 2020-05-26 02:04:00...               0.0120     0.01100
3  docker_003                    trace.self.jdbc.duration_p95        self           3  2020-05-26 02:04:00+0800                            1  2020-05-26 02:04:00+0800  [2020-05-26 02:04:00+0800, 2020-05-26 02:11:00...               0.0180     0.00700
4  docker_002                    trace.self.jdbc.duration_p95        self           2  2020-05-26 02:04:00+0800                            1  2020-05-26 02:04:00+0800  [2020-05-26 02:04:00+0800, 2020-05-26 02:11:00...               0.0140     0.00510, '')```
```

In [ ]:
# In[11]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[11]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz
import re
import numpy as np

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric and timezone-aware datetime (Asia/Shanghai UTC+8)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Identify groups relevant to docker_007
mask = (
    (trace_df['cmdb_id'] == 'docker_007') |
    trace_df['trace_name'].str.contains(r'\bto_docker_007\b', regex=True) |
    trace_df['trace_name'].str.contains(r'\bfrom_docker_007\b', regex=True)
)
groups_df = trace_df.loc[mask, ['cmdb_id', 'trace_name']].drop_duplicates().reset_index(drop=True)

# 2) Compute global_p95 per (cmdb_id, trace_name) using full series (no time filter)
global_p95_df = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(global_p95=lambda x: x.quantile(0.95)).reset_index()

# Merge to keep only selected groups with their global_p95
groups_thresh = groups_df.merge(global_p95_df, on=['cmdb_id','trace_name'], how='left')

# 3) Filter rows to incident window inclusive
tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
df_win = trace_df[(trace_df['ts_dt'] >= start) & (trace_df['ts_dt'] <= end)].copy()

# Keep only window rows for selected groups, and attach global_p95
df_win_sel = df_win.merge(groups_thresh, on=['cmdb_id','trace_name'], how='inner')

# 4) For each group compute requested aggregates
if df_win_sel.empty:
    docker_007_trace_overview = pd.DataFrame(columns=[
        'cmdb_id','trace_name','peer','count_window','count_high',
        'earliest_high_timestamp','max_value_in_window','global_p95'
    ])
    message = 'no highs for docker_007 in window'
else:
    # mark highs using precomputed global_p95
    df_win_sel['is_high'] = df_win_sel['value'] >= df_win_sel['global_p95']

    def agg_group(g):
        count_window = int(len(g))
        count_high = int(g['is_high'].sum())
        earliest_high = g.loc[g['is_high'], 'ts_dt'].min() if count_high>0 else pd.NaT
        earliest_high_str = earliest_high.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(earliest_high) else None
        max_in_win = g['value'].max()
        global_p95 = float(g['global_p95'].iloc[0]) if pd.notna(g['global_p95'].iloc[0]) else None
        return pd.Series({
            'count_window': count_window,
            'count_high': count_high,
            'earliest_high_timestamp': earliest_high_str,
            'max_value_in_window': max_in_win,
            'global_p95': global_p95
        })

    summary = df_win_sel.groupby(['cmdb_id','trace_name']).apply(agg_group).reset_index()

    # 5) parse peer from trace_name
    def parse_peer(name):
        m_to = re.search(r'\bto_([A-Za-z0-9_]+)', name)
        if m_to:
            return m_to.group(1)
        m_from = re.search(r'\bfrom_([A-Za-z0-9_]+)', name)
        if m_from:
            return m_from.group(1)
        return 'self'

    summary['peer'] = summary['trace_name'].apply(parse_peer)

    # Keep requested columns and order by earliest_high_timestamp ascending (NaT last)
    summary['earliest_dt'] = pd.to_datetime(summary['earliest_high_timestamp'], format='%Y-%m-%d %H:%M:%S%z', errors='coerce')
    summary = summary.sort_values(['earliest_dt', 'count_high'], ascending=[True, False])

    docker_007_trace_overview = summary[[
        'cmdb_id','trace_name','peer','count_window','count_high',
        'earliest_high_timestamp','max_value_in_window','global_p95'
    ]].head(20).reset_index(drop=True)

    # 6) If no groups have count_high>0, set message accordingly
    if docker_007_trace_overview['count_high'].sum() == 0:
        message = 'no highs for docker_007 in window'
        docker_007_trace_overview = pd.DataFrame(columns=[
            'cmdb_id','trace_name','peer','count_window','count_high',
            'earliest_high_timestamp','max_value_in_window','global_p95'
        ])
    else:
        message = ''

# Return the compact results (DataFrame and message)
docker_007_trace_overview, message

```
Out[12]:
```
Summary of docker_007 trace scan (window 2020-05-26 02:00–02:30 Asia/Shanghai):

- Scope & count: 20 (cmdb_id, trace_name) groups related to docker_007 were analyzed and returned.

- Broad observation: many groups show highs very early in the window (earliest_high_timestamp = 2020-05-26 02:00:00+0800) — several of these are error_rate or jdbc.duration metrics on inbound/outbound remoteprocess traces (e.g., trace.to_docker_007.remoteprocess.error_rate from docker_001 and docker_002, and corresponding from_docker_* entries on docker_007). For those groups count_high is 30 (every point in the window meets >= global_p95), often with global_p95 = 0.0 for error_rate metrics (artifact of constant zero series).

- Key actionable candidates:
  - docker_007 | trace.self.local.duration_p95 — count_high = 12 (highest among non-constant groups), earliest_high = 2020-05-26 02:03:00+0800, max_in_window = 0.0120, global_p95 = 0.01100. This is the strongest non-constant signal and should be prioritized.
  - docker_007 | trace.self.jdbc.duration_p95 and trace.self.jdbc.duration_mean / row_count variants — several show early highs (02:00–02:11) with some highs (2–30) and elevated max and p95 values; these point to JDBC/local processing anomalies on docker_007.
  - Remote peer timing: inbound remoteprocess.duration_mean/duration_p95 highs from docker_001 and docker_002 appear later (02:15–02:17) and may indicate downstream latency spikes involving docker_007’s peers (docker_001/docker_002 → docker_007).
  - Row_count metrics for docker_002/dockers → docker_007 show spikes at ~02:06 and ~02:11 (small counts of highs), which could reflect load bursts or sampling variance.

- Notes & caveats:
  - Some groups (especially error_rate traces) have global_p95 == 0.0, producing count_high = 30 because all values equal the threshold; treat those carefully as threshold artifacts.
  - The strongest non-artifact signal is docker_007 trace.self.local.duration_p95 (12 high points and multi-minute consecutive highs previously observed).

- Recommended next steps:
  1) Investigate docker_007 trace.self.local.duration_p95 time series in detail around 02:03–02:20 (highest concentration of highs).
  2) Correlate with docker_007 JDBC-related traces and row_count spikes (~02:04 and ~02:11).
  3) Inspect peer interactions at ~02:15–02:17 with docker_001/docker_002 to determine if network or downstream latency contributed.

If you want, I can (a) show the small time series snippets for the top candidate (docker_007 trace.self.local.duration_p95), or (b) compute longer consecutive-high segments and exact timestamps for any specific trace.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                         trace_name        peer  count_window  count_high   earliest_high_timestamp  max_value_in_window   global_p95
0   docker_001       trace.to_docker_007.remoteprocess.error_rate  docker_007            30          30  2020-05-26 02:00:00+0800             0.000000     0.000000
1   docker_002       trace.to_docker_007.remoteprocess.error_rate  docker_007            30          30  2020-05-26 02:00:00+0800             0.000000     0.000000
2   docker_007     trace.from_docker_001.remoteprocess.error_rate  docker_001            30          30  2020-05-26 02:00:00+0800             0.000000     0.000000
3   docker_007     trace.from_docker_002.remoteprocess.error_rate  docker_002            30          30  2020-05-26 02:00:00+0800             0.000000     0.000000
4   docker_007                       trace.self.jdbc.duration_p95        self            30          30  2020-05-26 02:00:00+0800             0.003000     0.003000
5   docker_007                         trace.self.jdbc.error_rate        self            30          30  2020-05-26 02:00:00+0800             0.000000     0.000000
6   docker_007                        trace.self.local.error_rate        self            30          30  2020-05-26 02:00:00+0800             0.000000     0.000000
7   docker_007                      trace.self.local.duration_p95        self            30          12  2020-05-26 02:03:00+0800             0.012000     0.011000
8   docker_001        trace.to_docker_007.remoteprocess.row_count  docker_007            30           2  2020-05-26 02:06:00+0800           296.000000   294.100000
9   docker_007      trace.from_docker_001.remoteprocess.row_count  docker_001            30           2  2020-05-26 02:06:00+0800           296.000000   294.100000
10  docker_007                      trace.self.jdbc.duration_mean        self            30           2  2020-05-26 02:10:00+0800             0.002700     0.002691
11  docker_002        trace.to_docker_007.remoteprocess.row_count  docker_007            30           2  2020-05-26 02:11:00+0800           311.000000   290.200000
12  docker_007      trace.from_docker_002.remoteprocess.row_count  docker_002            30           2  2020-05-26 02:11:00+0800           311.000000   290.200000
13  docker_007                          trace.self.jdbc.row_count        self            30           2  2020-05-26 02:11:00+0800          1515.000000  1420.800000
14  docker_007                         trace.self.local.row_count        self            30           2  2020-05-26 02:11:00+0800           911.000000   854.450000
15  docker_001    trace.to_docker_007.remoteprocess.duration_mean  docker_007            30           2  2020-05-26 02:15:00+0800             0.030746     0.030187
16  docker_007  trace.from_docker_001.remoteprocess.duration_mean  docker_001            30           2  2020-05-26 02:15:00+0800             0.030746     0.030187
17  docker_007                     trace.self.local.duration_mean        self            30           2  2020-05-26 02:15:00+0800             0.007473     0.007395
18  docker_001     trace.to_docker_007.remoteprocess.duration_p95  docker_007            30           2  2020-05-26 02:17:00+0800             0.066200     0.058535
19  docker_002    trace.to_docker_007.remoteprocess.duration_mean  docker_007            30           2  2020-05-26 02:17:00+0800             0.031068     0.030771, '')```
```

In [ ]:
# In[13]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric values and timezone-aware datetimes (Asia/Shanghai UTC+8)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'ts_dt' not in metric_df.columns:
    metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
else:
    metric_df['ts_dt'] = pd.to_datetime(metric_df['ts_dt']).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Select docker_007 rows
df_d7 = metric_df[metric_df['cmdb_id'] == 'docker_007'].copy()

# Prepare default outputs
docker_007_metric_anomalies = pd.DataFrame()
message = 'no metric highs for docker_007 in window'

if not df_d7.empty:
    # 2) Compute global P95 and P5 per kpi_name using full series for docker_007 (no time filter)
    global_thresh = df_d7.groupby('kpi_name')['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p5 =lambda x: x.quantile(0.05)
    ).reset_index()

    # Exclude constant-series where global_p95 == global_p5 (numeric tolerance)
    non_const = global_thresh[~np.isclose(global_thresh['global_p95'].astype(float),
                                         global_thresh['global_p5'].astype(float))].copy()

    if not non_const.empty:
        # 3) Filter docker_007 rows to incident window inclusive
        tz = pytz.timezone('Asia/Shanghai')
        start = pd.Timestamp('2020-05-26 02:00:00', tz=tz)
        end   = pd.Timestamp('2020-05-26 02:30:00', tz=tz)
        df_win = df_d7[(df_d7['ts_dt'] >= start) & (df_d7['ts_dt'] <= end)].copy()

        if not df_win.empty:
            # Keep only kpi_names that are non-constant
            df_win = df_win.merge(non_const, on='kpi_name', how='inner')

            # 4-5) Aggregate per kpi_name within window
            records = []
            for kpi, g in df_win.groupby('kpi_name'):
                g = g.sort_values('ts_dt').copy()
                count_window = int(len(g))
                # mark highs
                p95 = float(g['global_p95'].iloc[0])
                p5  = float(g['global_p5'].iloc[0])
                g['is_high'] = g['value'] >= p95
                count_high = int(g['is_high'].sum())
                if count_high > 0:
                    earliest_high_dt = g.loc[g['is_high'], 'ts_dt'].min()
                    earliest_high = earliest_high_dt.strftime('%Y-%m-%d %H:%M:%S%z')
                    # max/min in window
                    max_in_win = g['value'].max()
                    min_in_win = g['value'].min()
                    # find consecutive-high segments using only high timestamps
                    high_ts = g.loc[g['is_high'], 'ts_dt'].sort_values().reset_index(drop=True)
                    if len(high_ts) == 0:
                        longest_len = 0
                        longest_start = None
                    else:
                        # compute diffs between consecutive high timestamps
                        diffs = high_ts.diff().fillna(pd.Timedelta(seconds=999999))
                        # new run when diff != 1 minute
                        new_run = (diffs != pd.Timedelta(minutes=1))
                        run_id = new_run.cumsum()
                        runs = pd.DataFrame({'ts': high_ts, 'run_id': run_id})
                        run_stats = runs.groupby('run_id').agg(run_len=('ts','count'), run_start=('ts','min')).reset_index()
                        # select longest (tie-breaker: earliest start)
                        run_stats = run_stats.sort_values(['run_len','run_start'], ascending=[False, True])
                        longest_len = int(run_stats.iloc[0]['run_len'])
                        longest_start = run_stats.iloc[0]['run_start'].strftime('%Y-%m-%d %H:%M:%S%z')
                    # list of high timestamps (limit 10)
                    high_ts_list = high_ts.dt.strftime('%Y-%m-%d %H:%M:%S%z').tolist()[:10]
                else:
                    earliest_high = None
                    max_in_win = g['value'].max() if not g.empty else None
                    min_in_win = g['value'].min() if not g.empty else None
                    longest_len = 0
                    longest_start = None
                    high_ts_list = []

                records.append({
                    'kpi_name': kpi,
                    'count_window': count_window,
                    'count_high': count_high,
                    'earliest_high_timestamp': earliest_high,
                    'longest_segment_len_minutes': longest_len,
                    'longest_segment_start': longest_start,
                    'max_value_in_window': max_in_win,
                    'min_value_in_window': min_in_win,
                    'global_p95': p95,
                    'global_p5': p5,
                    'high_timestamps_list': high_ts_list
                })

            summary_df = pd.DataFrame.from_records(records)

            # Filter to rows with count_high > 0 (per instruction after excluding constants)
            summary_pos = summary_df[summary_df['count_high'] > 0].copy()

            if not summary_pos.empty:
                # 6) Order by count_high desc, then longest_segment_len desc, keep top 20
                summary_pos = summary_pos.sort_values(['count_high','longest_segment_len_minutes'], ascending=[False, False])
                docker_007_metric_anomalies = summary_pos[[
                    'kpi_name','count_window','count_high','earliest_high_timestamp',
                    'longest_segment_len_minutes','longest_segment_start',
                    'max_value_in_window','min_value_in_window','global_p95','global_p5'
                ]].head(20).reset_index(drop=True)
                message = ''
            else:
                docker_007_metric_anomalies = pd.DataFrame()
                message = 'no metric highs for docker_007 in window'
        else:
            docker_007_metric_anomalies = pd.DataFrame()
            message = 'no metric highs for docker_007 in window'
    else:
        # all kpis for docker_007 are constant-series
        docker_007_metric_anomalies = pd.DataFrame()
        message = 'no metric highs for docker_007 in window'
else:
    # no rows for docker_007
    docker_007_metric_anomalies = pd.DataFrame()
    message = 'no metric highs for docker_007 in window'

# Final compact output (store variables in kernel and return)
docker_007_metric_anomalies, message

```
Out[13]:
```
Result: No metric anomalies found for docker_007 in the incident window.

- Outcome: The analysis returned an empty DataFrame and message = "no metric highs for docker_007 in window".
- Meaning: After computing global P95/P5 per docker_007 KPI (using the full series), excluding constant-series (where global_p95 == global_p5), and checking points in the window 2020-05-26 02:00–02:30 (Asia/Shanghai), no KPI had any values ≥ its global P95. In other words, no metric-level highs were detected for docker_007 under the applied thresholding rules.
- Context & next steps (recommended):
  1. Note that trace analysis did show anomalies for docker_007 (e.g., trace.self.local.duration_p95). The absence of metric highs means the issue may be at trace / application-level rather than reflected in these metrics, or the metric thresholds/filters are too strict.
  2. If you want to continue: (a) relax thresholds (e.g., use P90 or an absolute delta), (b) exclude constant-series filtering temporarily, (c) inspect raw time series for high-resolution patterns around 02:03–02:20, and (d) correlate trace anomalies with relevant metrics (JVM, CPU, network) on docker_007 and its peers.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: []
Index: [], 'no metric highs for docker_007 in window')```
```